# Minha primeira LLM e conceitos basicos

Comecei estudando o básico de como um modelo de linguagem "lê" texto e terminei treinando um modelo meu, do zero, com um livro do Nietzsche.

Segui a aula do Prof. Marlon Ferrari no YouTube (LLM na Prática, com BERT) e fui adaptando do meu jeito.

In [11]:
%pip install pandas

import pandas as pd

### letra virando número

Rede neural não lê texto, só número. Então o primeiro passo é montar uma "cifra": cada letra ganha um número.

In [12]:
frase_base = 'o café da manhã esfriou na mesa'

# Quais simbolos únicos existem nessa frase?
simbolos = sorted(set(frase_base))
print('Símbolos encontrados:', simbolos)
print('Quantos símbolos o modelo conhece:', len(simbolos))

Símbolos encontrados: [' ', 'a', 'c', 'd', 'e', 'f', 'h', 'i', 'm', 'n', 'o', 'r', 's', 'u', 'ã', 'é']
Quantos símbolos o modelo conhece: 16


In [13]:
# Montando a "cifra": cada símbolo ganha um número
tabela_ida = {simbolo: posicao for posicao, simbolo in enumerate(simbolos)}
tabela_volta = {posicao: simbolo for posicao, simbolo in enumerate(simbolos)}

def codificar(texto):
    return [tabela_ida[s] for s in texto]

def decodificar(numeros):
    return ''.join(tabela_volta[n] for n in numeros)

In [14]:
teste1 = 'fome de manhã'
numeros = codificar(teste1)
print('Texto -> números:', numeros)
print('Números -> texto:', decodificar(numeros))

Texto -> números: [5, 10, 8, 4, 0, 3, 4, 0, 8, 1, 9, 6, 14]
Números -> texto: fome de manhã


### e se for palavra inteira?

Letra sozinha não quer dizer nada, então testei usar a palavra inteira (one-hot). Funciona, mas gera coluna demais e perde a ordem das palavras.

In [15]:
cardapio = pd.DataFrame({
    'pedido': [
        'quero um café com leite',
        'quero um suco de laranja',
        'hoje quero só um café',
    ]
})

# nesta etapa o modelo descobre todas as palavras diferentes e cria uma coluna para cada uma, 
# cada linha mostra quais palavras aquela frase usa

tabela_presenca = cardapio['pedido'].str.get_dummies(sep=' ')
tabela_presenca

,café,com,de,hoje,laranja,leite,quero,suco,só,um
0,1,1,0,0,0,1,1,0,0,1
1,0,0,1,0,1,0,1,1,0,1
2,1,0,0,1,0,0,1,0,1,1


### tokens, o meio-termo

É o que os LLMs de verdade usam: pedaços de palavra. Repara que o GPT-2 corta bem mais o português do que o inglês, porque foi treinado quase só com inglês.

In [16]:
%pip install tiktoken

In [17]:
# separar por letras: poucos símbolos, mas cada um sozinho não quer dizer nada
# separar por palavras: carregam significado, mas são símbolos demais, e qualquer palavra nova trava o modelo
import tiktoken

tokenizador_gpt = tiktoken.get_encoding('gpt2')

for frase in ['the coffee is hot', 'o café está quente', 'paralelepípedo']:
    ids = tokenizador_gpt.encode(frase)
    pedacos = [tokenizador_gpt.decode([i]) for i in ids]
    print(f'{frase!r}')
    print('   ids    :', ids)
    print('   pedaços:', pedacos)
    print('   tokens :', len(ids), '| letras:', len(frase))
    print()

'the coffee is hot'
   ids    : [1169, 6891, 318, 3024]
   pedaços: ['the', ' coffee', ' is', ' hot']
   tokens : 4 | letras: 17

'o café está quente'
   ids    : [78, 40304, 1556, 6557, 627, 21872]
   pedaços: ['o', ' café', ' est', 'á', ' qu', 'ente']
   tokens : 6 | letras: 18

'paralelepípedo'
   ids    : [1845, 1000, 293, 79, 8836, 9124, 78]
   pedaços: ['par', 'ale', 'le', 'p', 'í', 'ped', 'o']
   tokens : 7 | letras: 14



### o jogo do GPT e o jogo do BERT

O GPT aprende adivinhando a próxima palavra. O BERT aprende preenchendo uma lacuna no meio da frase, olhando pros dois lados. O modelo que eu treinei abaixo é do tipo BERT.

In [18]:
import random

frase_treino = 'o gato subiu no telhado para pegar o peixe'
palavras = frase_treino.split()

print('JOGO DO GPT: adivinhar a próxima palavra')
for fim in range(1, len(palavras)):
    contexto = palavras[:fim]
    resposta = palavras[fim]
    print(f'   vê: {" ".join(contexto):<40} -> deve chutar: {resposta}')

print()
print('JOGO DO BERT: preencher a lacuna')
for rodada in range(4):
    posicao = random.randrange(len(palavras))
    com_lacuna = palavras.copy()
    com_lacuna[posicao] = '<mask>'
    print(f'   vê: {" ".join(com_lacuna):<50} -> deve chutar: {palavras[posicao]}')

JOGO DO GPT: adivinhar a próxima palavra
   vê: o                                        -> deve chutar: gato
   vê: o gato                                   -> deve chutar: subiu
   vê: o gato subiu                             -> deve chutar: no
   vê: o gato subiu no                          -> deve chutar: telhado
   vê: o gato subiu no telhado                  -> deve chutar: para
   vê: o gato subiu no telhado para             -> deve chutar: pegar
   vê: o gato subiu no telhado para pegar       -> deve chutar: o
   vê: o gato subiu no telhado para pegar o     -> deve chutar: peixe

JOGO DO BERT: preencher a lacuna
   vê: <mask> gato subiu no telhado para pegar o peixe    -> deve chutar: o
   vê: o gato subiu no <mask> para pegar o peixe          -> deve chutar: telhado
   vê: o gato subiu no telhado para pegar o <mask>        -> deve chutar: peixe
   vê: o gato subiu no telhado para <mask> o peixe        -> deve chutar: pegar


---
## treinando a minha LLM
Daqui pra baixo eu treino um modelo do zero com o livro "Crepúsculo dos Ídolos" do Nietzsche.

In [19]:
%pip install "transformers[torch]" tokenizers

### 1. baixando o livro

In [20]:
import os
import urllib.request

os.makedirs('dados', exist_ok=True)
arquivo_livro = 'dados/crepusculo.txt'

url_livro = 'https://raw.githubusercontent.com/ferrarimarlon/NietzscheLLM/main/crepusculoDosIdolos.txt'
urllib.request.urlretrieve(url_livro, arquivo_livro)

with open(arquivo_livro, encoding='utf-8') as f:
    livro = f.read().replace('\r\n', '\n')

# regravo o arquivo sem o \r do windows pra não sujar o tokenizer
with open(arquivo_livro, 'w', encoding='utf-8') as f:
    f.write(livro)

print('caracteres no livro:', len(livro))
print('palavras no livro  :', len(livro.split()))
print()
print(livro[:400])

caracteres no livro: 156690
palavras no livro  : 26107

a ociosidade é mãe de toda psicologia. como? seria a psicologia um... vício?
o mais corajoso dentre nós dispõe apenas raramente da coragem de afirmar aquilo que sabe verdadeiramente...
para viver só é necessário ser um animal ou então um deus — afirma aristóteles. falta o terceiro caso: é necessário ser um e outro, é necessário ser — filósofo...
"toda verdade é simples." — não existe aí uma dupla 


### 2. treinando o meu próprio tokenizer

Lembra que o GPT-2 picotou o português todo? Então aqui eu crio um tokenizer que aprende os pedaços de palavra direto do livro.

É o mesmo tipo do GPT-2 (Byte-Level BPE): ele começa com bytes soltos e vai juntando os pares que mais aparecem até virar sílaba, palavra...

No vídeo o vocabulário é de 52 mil, mas o livro é pequeno (umas 25 mil palavras), então 8 mil já sobra.

In [21]:
from tokenizers import ByteLevelBPETokenizer

tokenizador_nietzsche = ByteLevelBPETokenizer()

tokenizador_nietzsche.train(
    files=[arquivo_livro],
    vocab_size=8_000,
    min_frequency=2,  # só vira token o que aparece pelo menos 2 vezes
    special_tokens=['<s>', '<pad>', '</s>', '<unk>', '<mask>'],
)

print('tokens no vocabulário:', tokenizador_nietzsche.get_vocab_size())

tokens no vocabulário: 5008


Os tokens especiais são tipo "marcadores": `<s>` é começo de texto, `</s>` é fim, `<pad>` é enchimento pra deixar tudo do mesmo tamanho, `<unk>` é "não sei" e `<mask>` é a lacuna que o modelo vai tentar adivinhar.

Agora a comparação que eu queria ver: o meu tokenizer contra o do GPT-2, na mesma frase.

In [22]:
frase_teste = 'a verdade é uma mulher e o filósofo não sabe disso'

ids_meu = tokenizador_nietzsche.encode(frase_teste).ids
ids_gpt = tokenizador_gpt.encode(frase_teste)

print('meu tokenizer:', len(ids_meu), 'tokens')
print('  ', [tokenizador_nietzsche.decode([i]) for i in ids_meu])
print()
print('GPT-2        :', len(ids_gpt), 'tokens')
print('  ', [tokenizador_gpt.decode([i]) for i in ids_gpt])

meu tokenizer: 11 tokens
   ['a', ' verdade', ' é', ' uma', ' mulher', ' e', ' o', ' filósofo', ' não', ' sabe', ' disso']

GPT-2        : 21 tokens
   ['a', ' ver', 'd', 'ade', ' é', ' u', 'ma', ' mul', 'her', ' e', ' o', ' fil', 'ó', 'so', 'fo', ' n', 'ão', ' s', 'abe', ' dis', 'so']


Mesmo com um vocabulário 6x menor, o meu tokenizer corta o português bem melhor, porque aprendeu com texto em português.

Agora salvo ele numa pasta. Saem dois arquivos: `vocab.json` (a tabela token → número) e `merges.txt` (as regras de junção que ele aprendeu).

In [23]:
pasta_modelo = 'meu_modelo'
os.makedirs(pasta_modelo, exist_ok=True)

tokenizador_nietzsche.save_model(pasta_modelo)
print(os.listdir(pasta_modelo))

['merges.txt', 'vocab.json']


### 3. carregando o tokenizer no formato do Hugging Face

A biblioteca `transformers` precisa do tokenizer num formato que ela conhece. Como o modelo vai ser um RoBERTa (a versão melhorada do BERT), uso o `RobertaTokenizer` lendo os arquivos que acabei de salvar.

In [24]:
from transformers import RobertaTokenizer

tokenizer = RobertaTokenizer.from_pretrained(pasta_modelo, model_max_length=128)

print(tokenizer('o homem é uma ponte')['input_ids'])
print('id do <mask>:', tokenizer.mask_token_id)

[0, 83, 463, 306, 339, 3946, 2]
id do <mask>: 4


### 4. montando o cérebro (o modelo RoBERTa)

Aqui eu defino o tamanho da rede. O do vídeo é bem maior (6 camadas, 768 neurônios por camada). Diminuí pra conseguir treinar no meu computador sem GPU:

- `hidden_size`: tamanho do vetor que representa cada token
- `num_hidden_layers`: quantas camadas de Transformer empilhadas
- `num_attention_heads`: quantas "cabeças de atenção" cada camada tem (cada uma presta atenção numa relação diferente entre as palavras)
- `max_position_embeddings`: até quantos tokens ele aguenta ler de uma vez (128 + 2 que o RoBERTa reserva)

O modelo nasce com pesos aleatórios. Ou seja, por enquanto ele só chuta.

In [25]:
from transformers import RobertaConfig, RobertaForMaskedLM

configuracao = RobertaConfig(
    vocab_size=len(tokenizer),
    max_position_embeddings=130,
    hidden_size=256,
    num_hidden_layers=4,
    num_attention_heads=4,
    intermediate_size=1024,
    type_vocab_size=1,
)

modelo = RobertaForMaskedLM(config=configuracao)
print(f'parâmetros: {modelo.num_parameters():,}'.replace(',', '.'))

parâmetros: 4.546.448


Pra ter noção: o GPT-3 tem 175 bilhões de parâmetros. O meu tem uns poucos milhões. É um bebê, mas é meu.

### 5. preparando os dados de treino

O vídeo lê o livro linha por linha. Só que nesse arquivo tem "linhas" com mais de 5 mil caracteres, e tudo que passa de 128 tokens é jogado fora. Achei desperdício.

Então fiz diferente: transformo o livro inteiro em uma fila de tokens e corto em blocos de 128. Assim o modelo vê o livro todo.

In [26]:
ids_livro = tokenizer(livro, add_special_tokens=False)['input_ids']
print('tokens no livro inteiro:', len(ids_livro))

tamanho_bloco = 126  # +2 dos marcadores <s> e </s> = 128

blocos = []
for inicio in range(0, len(ids_livro) - tamanho_bloco + 1, tamanho_bloco):
    pedaco = ids_livro[inicio:inicio + tamanho_bloco]
    blocos.append({'input_ids': [tokenizer.bos_token_id] + pedaco + [tokenizer.eos_token_id]})

print('blocos de treino:', len(blocos))
print()
print('exemplo de bloco:')
print(tokenizer.decode(blocos[3]['input_ids']))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (35570 > 128). Running this sequence through the model will result in indexing errors


tokens no livro inteiro: 35570
blocos de treino: 282

exemplo de bloco:
<s> exemplo — são menos compreendidos que aqueles que são conformes sua época, mas escutamo-los melhor. que eu me exprima mais precisamente ainda: jamais somos compreendidos — e é disso que advém nossa autoridade...
entre mulheres. — "a verdade? oh, não conheces a verdade! não é ela um atentado contra nosso pudor?"
eis um artista como os aprecio. é modesto em suas necessidades: requer, em suma, somente duas coisas. seu pão e sua arte — panem et circen...
aquele que não sabe dispor sua vontade nas coisas quer ao menos atribuir-</s>


### 6. o "professor" que esconde palavras

O `DataCollatorForLanguageModeling` monta os lotes de treino e faz o jogo do BERT sozinho: em cada bloco ele esconde 15% dos tokens com `<mask>` e o modelo tem que adivinhar o que estava ali.

O legal é que a cada rodada ele sorteia lacunas diferentes, então o mesmo trecho vira um exercício novo toda vez.

In [27]:
from transformers import DataCollatorForLanguageModeling

coletor = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15,
)

# só pra ver o que ele faz com um bloco
amostra = coletor([blocos[3]])
print(tokenizer.decode(amostra['input_ids'][0]))

<s> exemplo — são menos compreendidos que aqueles que<mask> conformes sua época, mas escutamo-<mask> melhor.ma eu me<mask>rima mais precisamente ainda:<mask> somos compreendidos — e<mask><mask> que advém nossa autoridade...
entre mulheres. — "reza verdade? oh<mask> nãodoes a verdade! não<mask> ela um ate<mask> contra<mask><mask>dor?"
eis um<mask> como ososscio. é modesto em suas necessidades: requer, em suma, somente duas coisas. seu pão e sua arte — panem penitênciat circen...
aquele<mask><mask> sabe dispor sua vontade nas coisas quer<mask> menos atribuir<mask></s>


### 7. treinando

Cada época é o modelo lendo o livro inteiro uma vez. O vídeo usa 1200 épocas, o que no CPU levaria uma eternidade. Coloquei 100.

Se tiver GPU (ou rodar no Colab), o `Trainer` usa sozinho e dá pra subir esse número. No CPU leva alguns minutos, então é hora do café.

O número pra acompanhar é o `loss`: é o tamanho do erro. Ele tem que ir caindo.

In [28]:
import torch
from transformers import Trainer, TrainingArguments

print('GPU disponível?', torch.cuda.is_available())

EPOCAS = 100

argumentos = TrainingArguments(
    output_dir=os.path.join(pasta_modelo, 'checkpoints'),
    num_train_epochs=EPOCAS,
    per_device_train_batch_size=16,
    learning_rate=5e-4,
    warmup_steps=100,
    logging_steps=100,
    save_strategy='no',
    report_to='none',
    seed=42,
)

treinador = Trainer(
    model=modelo,
    args=argumentos,
    data_collator=coletor,
    train_dataset=blocos,
)

GPU disponível? False


In [30]:
treinador.train()

Step,Training Loss
100,5.597916
200,5.508694
300,5.338083
400,5.213588
500,5.074733
600,4.914734
700,4.775221
800,4.671371
900,4.504501
1000,4.416118


TrainOutput(global_step=1800, training_loss=4.564246910942925, metrics={'train_runtime': 847.3844, 'train_samples_per_second': 33.279, 'train_steps_per_second': 2.124, 'total_flos': 73669902458880.0, 'train_loss': 4.564246910942925, 'epoch': 100.0})

O loss ainda tava caindo quando acabou (de 5.6 pra 3.8), ou seja, o modelo ainda tava aprendendo. Então treinei mais 200 épocas em cima do mesmo modelo, de onde parou.

In [35]:
import torch
from transformers import Trainer, TrainingArguments

print('GPU disponível?', torch.cuda.is_available())


# continuo treinando o mesmo modelo, de onde parou
argumentos_extra = TrainingArguments(
    output_dir=os.path.join(pasta_modelo, 'checkpoints'),
    num_train_epochs=200,
    per_device_train_batch_size=16,
    learning_rate=3e-4,
    logging_steps=200,
    save_strategy='no',
    report_to='none',
    seed=7,
)

treinador = Trainer(model=modelo, args=argumentos_extra, data_collator=coletor, train_dataset=blocos)
treinador.train()

GPU disponível? False


Step,Training Loss
200,3.834951
400,3.613369
600,3.387481
800,3.141157
1000,2.919557
1200,2.722592
1400,2.516456
1600,2.325090
1800,2.148703
2000,2.009712


TrainOutput(global_step=3600, training_loss=2.281445388793945, metrics={'train_runtime': 1479.0234, 'train_samples_per_second': 38.133, 'train_steps_per_second': 2.434, 'total_flos': 139945520332800.0, 'train_loss': 2.281445388793945, 'epoch': 200.0})

In [37]:
# salvando o modelo treinado junto com o tokenizer
treinador.save_model(pasta_modelo)
tokenizer.save_pretrained(pasta_modelo)
print(os.listdir(pasta_modelo))

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 95.23it/s]

['checkpoints', 'config.json', 'merges.txt', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin', 'vocab.json']


### 8. a hora da verdade

Agora eu escrevo uma frase com `<mask>` e o modelo me diz o que ele acha que vai ali, com a porcentagem de certeza.

Primeiro tentei com o `pipeline` do Hugging Face, mas ele se enrolava com o espaço antes do `<mask>` e o modelo ficava chutando pedaço de palavra. Aí montei a frase na mão, do mesmo jeito que o modelo viu no treino.

Detalhe: o livro está todo em minúsculo, então escrevo as frases em minúsculo também.

In [41]:
import torch

modelo_treinado = treinador.model
modelo_treinado.eval()  # modo de teste, desliga as partes que só servem pro treino

def perguntar(frase, quantos=5):
    print(frase)
    antes, depois = frase.split('<mask>')

    # monto a frase na mão: <s> + texto antes + <mask> + texto depois + </s>
    ids = ([tokenizer.bos_token_id]
           + tokenizer.encode(antes.rstrip(), add_special_tokens=False)
           + [tokenizer.mask_token_id]
           + tokenizer.encode(depois, add_special_tokens=False)
           + [tokenizer.eos_token_id])
    posicao_lacuna = ids.index(tokenizer.mask_token_id)

    with torch.no_grad():
        entrada = torch.tensor([ids], device=modelo_treinado.device)
        notas = modelo_treinado(entrada).logits[0, posicao_lacuna]

    chances = notas.softmax(dim=-1)
    melhores = chances.topk(quantos)
    for chance, token_id in zip(melhores.values, melhores.indices):
        palavra = tokenizer.decode([int(token_id)]).strip()
        print(f'   {palavra:<15} {chance.item():.1%}')
    print()

In [42]:
# essa frase está no livro, então é um teste de memória
perguntar('o que não me faz morrer me torna mais <mask>.')

# essas eu inventei
perguntar('a moral é uma invenção dos <mask>.')
perguntar('o filósofo não tem <mask>.')
perguntar('deus é um <mask> do homem.')

o que não me faz morrer me torna mais <mask>.
   o               19.5%
   —               7.1%
   ele             4.4%
   faz             4.2%
   necessário      3.6%

a moral é uma invenção dos <mask>.
   crimin          20.1%
   organizadora    9.9%
   resfria         4.2%
   xix             3.7%
   boa             3.3%

o filósofo não tem <mask>.
   se              13.0%
   liberais        4.9%
   homem           4.8%
   ge              4.1%
   mesmo           3.5%

deus é um <mask> do homem.
   uma             15.3%
   a               8.8%
   não             3.7%
   enquanto        3.5%
   contra          3.4%



Brinca à vontade aqui embaixo, é só trocar a frase:

In [ ]:
perguntar('a verdade é <mask>.')

---
### o que eu aprendi com isso

O modelo não "entende" Nietzsche. Ele aprendeu quais pedaços de palavra costumam aparecer perto de quais, lendo o mesmo livro 300 vezes.

E ele ainda erra bastante. Na frase que está no livro ("o que não me faz morrer me torna mais forte") ele nem chegou perto do "forte".

Mas em outras dá pra ver que ele pegou o clima do livro. Pra "a moral é uma invenção dos ___" o primeiro chute foi "crimin", de criminoso, palavra que aparece várias vezes no livro. Ele só consegue chutar um token por vez, por isso veio cortado.

Tudo isso com 4,5 milhões de parâmetros, um livro de 26 mil palavras e uns 40 minutos de treino no meu PC, sem GPU. Os modelos grandes fazem a mesma coisa, só que com bilhões de parâmetros e uma quantidade absurda de texto.

O que mais me pegou foi o tokenizer. Antes eu achava que o modelo lia palavra por palavra. Na verdade ele lê pedaços, e quem decide esses pedaços é treinado antes do modelo. Até o espaço antes de uma palavra faz diferença, como eu descobri apanhando no teste.

Próximo passo: treinar com mais livros e testar um modelo tipo GPT, que gera texto em vez de só preencher lacuna.

Aula que eu segui: [LLM na Prática, Prof. Marlon Ferrari](https://www.youtube.com/watch?v=LphBQB1amrw)